# 4. Deploy on New Data

This notebook applies the locked final pipeline to "new_data.csv" and generate a prioritized review list.

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import os
import pandas as pd

from src.utils import load_pipeline, load_new, prioritize_new_data

## Load the locked pipeline and new data
We load the saved final pipeline and the newly released dataset.

In [3]:
pipe = load_pipeline("../models/final_model.joblib")


new_df = load_new("../data/new_data.csv")

print("New data shape:", new_df.shape)
new_df.head()

New data shape: (2000, 17)


,id,day,event_type,category,region,device,account_age_days,num_prev_listings,prev_reports_30d,verification_level,price,num_images,message_length,contains_off_platform,urgency_words,payment_attempt,time_to_first_response_min
0,0,1,message_send,bikes,NaN,android,46.6,2,0,1,149.83,9,96,0,0,0,37.2
1,1,2,ad_post,fashion,rural,web,131.6,3,1,0,123.22,4,108,0,0,0,32.5
2,2,2,message_send,bikes,NaN,android,135.0,2,0,1,223.22,2,227,0,1,1,11.3
3,3,5,ad_post,phones,rural,ios,81.3,5,0,1,1152.26,8,72,1,1,1,2.8
4,4,3,message_send,furniture,metro,web,33.2,3,0,0,100.22,2,210,0,0,0,10.2


## Generate prioritized review list
We apply the locked pipeline to the new data and rank all events by predicted risk score.

Based on our earlier Operations requirement, we use Top-5% as the prioritization rule.

In [4]:
prioritized = prioritize_new_data(pipe, new_df, top_frac=0.05)

prioritized.head()


,id,risk_score
1092,1092,0.796287
1398,1398,0.750542
1547,1547,0.669184
1130,1130,0.666560
1831,1831,0.627240


## Save prioritized cases
The prioritized list is exported so it can be used by the review team.

In [5]:
os.makedirs("../outputs", exist_ok=True)

prioritized.to_csv("../outputs/prioritized_cases.csv", index=False)

print("Saved to ../outputs/prioritized_cases.csv")

Saved to ../outputs/prioritized_cases.csv


## Deployment reflection

The final pipeline was locked before "new_data.csv" was released.  
No retraining, no new feature engineering changes or no hyperparameter changes were made after locking.

Only the prioritization rule (Top-5%) is applied to decide which cases should be reviewed first.